# YOLOv11 Training Pipeline (Google Colab)
Notebook ini digunakan untuk training model YOLOv11 menggunakan GPU dari Google Colab.

In [1]:
# Cek GPU yang tersedia
!nvidia-smi

Fri Apr 24 13:36:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 581.80                 Driver Version: 581.80         CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   45C    P8              3W /   95W |      11MiB /   4096MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# Install ultralytics 
!pip install -q ultralytics roboflow 

import torch 
print(f"CUDA Availabel: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import os 
from ultralytics import YOLO 
from google.colab import drive 
print("Ultralytics version:", __import__('ultralytics').__version__)

In [ ]:
# Mount Drive untuk menyimpan hasil training 
drive.mount('/content/drive')

# Membuat folder output di Drive 
DRIVE_OUTPUT =  '/content/drive/MyDrive/yolov11_result'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)

In [ ]:
from roboflow import Roboflow

# sesuaikan dengan info dari Roboflow user 
rf = Roboflow(api_key ="YOUR_API_KEY")
project = rf.workspace("YOUR_WORKSPACE").project("YOUR_PROJECT")
version = project.version(YOUR_VERSION_NUMBER)
dataset = version.download("yolov8") # format YOLO ini tetap kompatibel dengan v11 

data_yaml= os.path.join(dataset.location, 'data.yaml')
print(f"Dataset location: {dataset.location}")
print(f"data.yaml: {data_yaml}")

In [ ]:
# Verifikasi dataset 
for root, dirs, files in os.walk(dataset.location):
    level = root.replace(dataset.location, ' ').count(os.sep)
    indent = '  ' * level 
    print(f'{indent}{os.path.basename(root)}/')
    subindent = '  ' * (level + 1)
    for f in files[:10]: 
        print(f'{subindent}{f}')

## Configure Training 
Ubah parameter sesuai dengan kebutuhan 

In [ ]:
# KONFIGURASI TRAINING

# Model 
# yolo11n.pt = Nano (Tercepat)
# yolo11s.pt = Small 
# yolo11m.pt = Medium 
# yolo11l.pt = Large
# yolo11x.pt = Extra Large (Paling akurat)
MODEL_NAME = 'yolo11.pt' # sesuaikan sendiri 

# Training 
EPOCHS = 64
BATCH_SIZE = 16 
IMG_SIZE = 640
PATIENCE = 20       # Early Stopping 
DEVICE = 0          # 0 = GPU PERTAMA

# Nama Project 
PROJECT_NAME = 'runs/yolov11'
EXPERIMENT_NAME = 'train_colab'

In [ ]:
# Load pretrained model 
model = YOLO(MODEL_NAME)

# Jalankan Training
result = model.train(
    data = data_yaml,
    epochs = EPOCHS,
    batch = BATCH_SIZE,
    imgsz = IMG_SIZE,
    patience = PATIENCE, 
    device = DEVICE, 
    project = PROJECT_NAME,
    name = EXPERIMENT_NAME, 
    plots = True, 
    save = True, 
    val = True,
)
print(result)

In [ ]:
from IPython.display import Image, display

# Tampilkan grafik hasil training 
result_dir = f'{PROJECT_NAME}/{EXPERIMENT_NAME}'
for img_name in ['result.png', 'confusion_matrix.png', 'val_batch0_pred.jpg']:
    img_path = os.path.join(result_dir, img_name)
    if os.path.exist(img_path):
        print(f'\n--- {img_name} ---')
        display(Image(filename=img_path, width=800))

In [ ]:
# Validation pada validation set 
best_model = YOLO(f'{result_dir}/weight/best.pt')
metrics = best_model.val(data=data_yaml, split='val', device=DEVICE)

print(f"\nmAP@50   : {metrics.box.map50:.4f}")
print(f"\mAP@50-95 : {metrics.box.map:.4f}")
print(f"\Precision : {metrics.box.map:.4f}")
print(f"\Recal     : {metrics.box.map:.4f}")